In [1]:
import sys
USER_HOME='/ihome/rboyce/rdb20/'
sys.path.append(USER_HOME + '.conda/envs/rnn-and-transformer/lib/python3.11/site-packages/')

In [2]:
import os
import scipy as sp
import sklearn as skl
import matplotlib.pyplot as plt
from copy import deepcopy

#import warnings
#warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.metrics import roc_curve, auc, f1_score, accuracy_score, precision_recall_curve
from sklearn.preprocessing import MinMaxScaler

import numpy as np
from numpy.random import seed

import pandas as pd
from pandas.api.types import CategoricalDtype

import random

from IPython.display import HTML

In [3]:
# Verify PyTorch installation and version
print("PyTorch version:", torch.__version__)

# Check if CUDA is available
if torch.cuda.is_available():
    print("CUDA is available. PyTorch can use the GPU.")
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
else:
    print("CUDA is not available. PyTorch will use the CPU.")

# Get the current device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Current device in use:", device)

# Run a quick tensor operation on GPU (if available)
x = torch.rand(3, 3).to(device)
print("Tensor operation result on", device, ":\n", x)

PyTorch version: 2.5.1
CUDA is available. PyTorch can use the GPU.
GPU Name: NVIDIA GeForce GTX TITAN X
CUDA Version: 12.1
Current device in use: cuda
Tensor operation result on cuda :
 tensor([[0.0537, 0.6885, 0.5146],
        [0.7857, 0.7113, 0.7856],
        [0.6370, 0.2358, 0.0373]], device='cuda:0')


In [4]:
print("python version: ", sys.version)
print("pandas version: ", pd.__version__)
print('numpy version: ', np.__version__)
print('scipy version: ', sp.__version__)
print('sklearn version: ', skl.__version__)
print('torch version: ', torch.__version__)

python version:  3.11.5 (main, Sep 11 2023, 13:54:46) [GCC 11.2.0]
pandas version:  2.2.2
numpy version:  1.26.3
scipy version:  1.11.4
sklearn version:  1.5.2
torch version:  2.5.1


In [5]:
# Set CUDA_LAUNCH_BLOCKING to 1
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# Verify the setting
print("CUDA_LAUNCH_BLOCKING:", os.environ.get("CUDA_LAUNCH_BLOCKING"))

CUDA_LAUNCH_BLOCKING: 1


In [6]:
# Define a simple RNN to confirm that the environment works
rnn = nn.RNN(
    input_size=101,  # Feature size
    hidden_size=32,  # Hidden state size
    num_layers=1,    # Single RNN layer
    batch_first=True
)

# Check RNN parameters
print("RNN Parameters:")
for name, param in rnn.named_parameters():
    print(f"{name}: {param.shape}, requires_grad={param.requires_grad}")

# Test forward pass with dummy data
dummy_input = torch.randn(10, 20, 101)  # (batch_size, sequence_length, feature_size)
output, hidden = rnn(dummy_input)
print("Output shape:", output.shape)
print("Hidden shape:", hidden.shape)

#Expected Output:
# Parameters:
# weight_ih_l0: [32, 101]
# weight_hh_l0: [32, 32]
# bias_ih_l0: [32]
# bias_hh_l0: [32]
# Output:
# Shape: [10, 20, 32] (batch_size, sequence_length, hidden_size)
# Hidden state: [1, 10, 32] (num_layers, batch_size, hidden_size)

RNN Parameters:
weight_ih_l0: torch.Size([32, 101]), requires_grad=True
weight_hh_l0: torch.Size([32, 32]), requires_grad=True
bias_ih_l0: torch.Size([32]), requires_grad=True
bias_hh_l0: torch.Size([32]), requires_grad=True
Output shape: torch.Size([10, 20, 32])
Hidden shape: torch.Size([1, 10, 32])


### Load data

In [7]:
excel_file = r'full_set_copy.xls'
df = pd.DataFrame(pd.read_excel(excel_file))
print(df.shape)

(13142, 208)


In [8]:
df.head(n=10)

,patient-id,episode-start-date,episode-end-date,stay-start-date,stay-end-date,predictor-date,projected-date,current-stay-days,cumulative-days-in-facility,cms-long-stay,...,psych_sdd_3,psych_sdd_4,psych_sdd_5,psych_sdd_final,atc_1,atc_2,atc_3,atc_4,atc_5,number_of_behavior_changes
0,10353,2013-07-19 00:00:00,2013-08-19 00:00:00,2013-07-19 00:00:00,2013-08-19 00:00:00,2013-07-26 00:00:00,2013-08-19 00:00:00,7,7,No,...,1.333333,0.0,0,4.000000,N06AX21,N05BA12,N06AX05,NaN,NaN,0
1,10364,2013-08-07 00:00:00,2013-08-27 00:00:00,2013-08-07 00:00:00,2013-08-27 00:00:00,2013-08-14 00:00:00,2013-08-27 00:00:00,7,7,No,...,0.000000,0.0,0,0.166667,N05AH04,NaN,NaN,NaN,NaN,0
2,10379,2013-09-24 00:00:00,2013-10-15 00:00:00,2013-09-24 00:00:00,2013-10-15 00:00:00,2013-10-01 00:00:00,2013-10-15 00:00:00,7,7,No,...,0.000000,0.0,0,2.333333,N06AB04,N06AX05,NaN,NaN,NaN,0
3,10385,2013-10-12 00:00:00,2013-11-01 00:00:00,2013-10-12 00:00:00,2013-11-01 00:00:00,2013-10-19 00:00:00,2013-11-01 00:00:00,7,7,No,...,0.000000,0.0,0,3.000000,N06AB05,N06AX11,NaN,NaN,NaN,0
4,10386,2013-10-18 00:00:00,2013-10-29 00:00:00,2013-10-18 00:00:00,2013-10-29 00:00:00,2013-10-25 00:00:00,2013-10-29 00:00:00,7,7,No,...,0.000000,0.0,0,3.500000,N06AB06,N05AH03,NaN,NaN,NaN,0
5,5000,2012-04-25 00:00:00,2012-05-08 00:00:00,2012-04-25 00:00:00,2012-05-08 00:00:00,2012-05-02 00:00:00,2012-05-08 00:00:00,7,7,No,...,0.000000,0.0,0,0.500000,N06AX16,NaN,NaN,NaN,NaN,0
6,5003,2011-08-02 00:00:00,2011-08-24 00:00:00,2011-08-02 00:00:00,2011-08-24 00:00:00,2011-08-09 00:00:00,2011-08-24 00:00:00,7,7,No,...,0.000000,0.0,0,0.625000,N05AX08,N06AB04,NaN,NaN,NaN,0
7,5003,2011-08-02 00:00:00,2011-09-29 00:00:00,2011-09-05 00:00:00,2011-09-29 00:00:00,2011-09-12 00:00:00,2011-09-29 00:00:00,7,41,No,...,0.000000,0.0,0,1.125000,N05AX08,N06AB04,NaN,NaN,NaN,0
8,5007,2012-10-21 00:00:00,2012-11-15 00:00:00,2012-10-27 00:00:00,2012-11-15 00:00:00,2012-11-03 00:00:00,2012-11-15 00:00:00,7,13,No,...,0.000000,0.0,0,0.000000,NaN,NaN,NaN,NaN,NaN,0
9,5011,2012-06-12 00:00:00,2012-07-01 00:00:00,2012-06-12 00:00:00,2012-07-01 00:00:00,2012-06-19 00:00:00,2012-07-01 00:00:00,7,7,No,...,0.000000,0.0,0,0.000000,NaN,NaN,NaN,NaN,NaN,0


In [9]:
df.columns.tolist()

['patient-id',
 'episode-start-date',
 'episode-end-date',
 'stay-start-date',
 'stay-end-date',
 'predictor-date',
 'projected-date',
 'current-stay-days',
 'cumulative-days-in-facility',
 'cms-long-stay',
 'mds-stay-trans-short-to-long',
 'age',
 'gender',
 'race',
 'marital-status',
 'mds-entered-from',
 'facility',
 'mds-antianxiety-medication',
 'mds-antidepressant-medication',
 'mds-antipsychotic-medication',
 'mds-antibiotic-medication',
 'mds-anticoagulant-medication',
 'mds-diuretic-medication',
 'mds-hypnotic-medication',
 'mds-behavioral-symptoms',
 'mds-behavioral-symptoms-to-others',
 'mds-bims-summary-ranking',
 'mds-cognitive-scale',
 'mds-delirium-scale',
 'mds-dehydrated',
 'mds-depression',
 'mds-fever',
 'mds-impaired-mobility',
 'mds-impaired-transfer',
 'mds-internal-bleeding',
 'mds-pain-last-five-days',
 'mds-pain-freq-last-five-days',
 'mds-pain-non-verbal',
 'mds-pain-medication',
 'mds-received-pain-tx-non-pharm',
 'mds-received-prn-pain-medication',
 'mds-pai

In [10]:
# Drop features
f_drop = ["Blank-1", "ws-sudden-stop", "ws-sudden-stop-drugs", "ws-prolonged-psychotropics", "mds-toilet-prgm-atmptd"] + \
         ['Deleted-%d' % i for i in range(1, 13)] + ['Psychotropic-%d' % i for i in range(1, 6)] + \
         ['Psychotropic-%d-average-daily-dose' % i for i in range(1, 6)] + ["Entry-discharge-type"]

# Missing value means not relevant
f_one_hot = ["Mds-fall-2-6-months-to-admission", "Mds-fall-last-month-to-admission", "mds-pain-last-five-days", \
             "mds-delirium-scale", "mds-long-term-memory-ok", "mds-short-term-memory-ok", "mds-staff-assess-pain", \
             "recent-start-other-fall-risk-rx"] + ["mds-pressure-ulcer-stage-%d" % i for i in range(1, 5)]
cate_one_hot = [['Yes', 'No', 'Unable to answer']] * 3 + [['Yes', 'No']] * 9

f_period = [("Psychotropic-%d-Start-date" % i, 'Psychotropic-%d-End-date' % i, 'Psychotropic-%d-Period' % i) for i in range(1, 6)]

f_label = ['mds-cognitive-scale', 'mds-pain-freq-last-five-days', 'mds-pain-intensity']
dic_label = [{'Independent': 0, 'Modified Independence': 1, 'Moderately Impaired': 2, 'Severely Impaired': 3}, \
             {'Continuous': 4, 'Frequent': 3, 'Occasional': 2, 'Rare': 1, 'Unable to respond': 0}, \
             {'None': 0, 'Mild': 1, 'Moderate': 2, 'Severe': 3, 'Very severe, horrible': 4}]

f_transform = [('mds-cognitive-scale', 3, 0), ('mds-pain-freq-last-five-days', 4, 0), ('mds-pain-intensity', 4, 0)] + \
              [('Psychotropic-%d-Period' % i, 0, 0) for i in range(1, 6)]

In [11]:
# Encode features (originally) with no missing values
f_drop += ["predictor-date", "projected-date", "PPS-assessment", "Federal-assessment"]

f_period += [('episode-start-date', 'episode-end-date', 'episode-period'),\
             ('stay-start-date', 'stay-end-date', 'stay-period')]

f_label += ["cms-long-stay", "mds-stay-trans-short-to-long", "mds-antianxiety-medication", \
            "mds-antidepressant-medication", "mds-antipsychotic-medication", "mds-antibiotic-medication",
            "mds-anticoagulant-medication", "mds-diuretic-medication", \
            "mds-hypnotic-medication", "mds-behavioral-symptoms", "mds-behavioral-symptoms-to-others", \
            "mds-dehydrated", "mds-depression", "mds-fever", "mds-impaired-mobility", "mds-impaired-transfer", \
            "mds-internal-bleeding", "mds-malnutrition", "mds-no-problem-conditions", "mds-vomiting", \
            "mds-impaired-walk-in-room", "mds-impaired-walk-in-corridor", "mds-impaired-locomot-unit", \
            "mds-impaired-locomot-other", "ws-antibiotic-anticoag-coexposure", \
            "ws-psychotropic-with-not-ordered-weight-loss", \
            "ws-diuretic-adl", "ws-tramadol-antidepressant-coexposure"] + ["mds-pain-non-verbal"]
dic_label += [{'Yes': 1, 'No': 0}] * 28 + [{'None/Mild':0,'Moderate/Severe':1}]

f_one_hot += ["race", "facility", "gender", "mds-bims-summary-ranking", "ws-meclizine-psych-coexposure"]
cate_one_hot += [['White', 'Black', 'Other'], \
                 ['Sugar Creek', 'Heritage Place', 'Canterbury Place', 'Senaca Place', 'Cranberry Place'], \
                 ['Female', 'Male'], ['Intact or Moderately Intact', 'Moderate Impairment'], \
                 ['No', 'meclAndPsychStartSameR', 'meclPrecedesPsychR', 'psychPrecedesMeclR', 'meclAndPsychOlderStart']]

In [12]:
f_drop += ["min_effective_dose_%d" % i for i in range(1, 6)] + \
          ["Psychotropic-%d-average-daily-dose" % i for i in range(1, 6)] + \
          ["psych_sdd_%d" % i for i in range(1, 6)]

In [13]:
df['patient-id'] = df['patient-id'].astype('str')

In [14]:
list_pid = ['01c9bd28cd38934a6b6598c2f0c595e2','022c3df43beb87426fdc9e3aece863ed','028fedbb573c6bfa76fbde3cfef2aeee',
            '03af5cb05c302b9a8c2ccae1c91da4d0','0461c03862af4e50ce159a73f004b001','050e80396569549c813bd86878395edd',
            '059e7c93e64ac77e36a2e5837e48be77','05f01f0d1a7272b250d63cb579b1daf2','066a29d308fa4d898102a3e9f647de6f',
            '066ce145d5b174af066254e5be1585e2','0912055f2195755d7bef28609d049dd0','0f79f492f4c8473c31201ccd41f102b8',
            '125560598c5696e07f71c3f4da381d2f','135ceeb77bd08c60a947f74b9fd83d00','159cf0cc2acd9e91a05a523c7a152329',
            '16f552deb3c2ba571d073738cb554126','19632eeb65376617bd612503132285e2','1d46d1f923cfa7e3a7b23a737637981b',
            '1e15c99576756e3a1823a5f6aedd7f18','1fa6c165ef356f3252f06e7ca18c50fe','2085480e932df230b019761f6174200a',
            '2113f02d0df3a6963ff61e16bc4ad767','243bca59454a35fd31a80856898f4f08','28be643baa0fb455c9c311646f0041b8',
            '2a6db0f42c1d5d1c4f1342c5291b6c01','2b57427d702a8c4e375cd02fdd0cf387','2b6732124cae3953adecd89836537beb',
            '2d71fe90931c6e12ef800c4cba5377e3','3157f5db65d2f725c739a7ef98ec8b4e','32ac39841c469a47d86eb43abf35f3d5',
            '33eccc93bc55ba7619a6d1e333baf1f5','343df02f2dfc3c1ad6428e7e1c246585','37f069da79c11e4e9c01794b38bfa39f',
            '394f22d67bf57e6bfeebcf21242b38a5','3cff678dc116ea0c96bbc918057bc7c5','3d3fddbdfd3583a6d9d2c8e7f30c3cfb',
            '3d73b4edb7027ac1a8afd46ecc91e010','3df5c258d2032361615b2f9855bf554f','3e1f2a5ed144a2564c9857a530d25435',
            '4125a264d97877a301892fdd0f40431e','419328961cb5cf6fbaeb4178d2b77e46','4272f93c23b65e2c2a76a1daf8d40c43',
            '4b3fb60a20ef687a3f7c7d298467ee98','4cb9543864cb8cbe3e4bd66b0928b3f3','4cf06d285a6c3ca4591603c22536e2f0',
            '4d0739e22fc57d8cee62ad1d07ae1dd8','4e9a52901f6c1c38f0651378a4b60c61','50c824b1d52dc56df894d991f2ec46a8',
            '528ba7513e8a789dbb47ef90ccd861ac','58cc1f6d463a9034a9cf51c1f7e50804','5914c3856eba88d19500ce0cafd35ef1',
            '59545dcc4048310c4b8f09269ac7ce07','5a5aaba2d783818513a3cf640a18720a','5e3e1d07b47dac2ced35258ae59ff686',
            '602dd3b3ef35189913b236748df90361','606aa687675a9ea23d9b9a7329e1ca89','61422cfa73ea89c6d0b0268b6c89e492',
            '617487bcc3dc2194fd094ea86e0e49e9','6219b2510e1b28eb0a7f48c3f7fdb52c','63f1056a29e7b12278e41d0f13f59de6',
            '64badf6afcc2b2e7b2288c17d74af035','65d7aa76dc841c84dfa376ed2bff6d65','6879012f2cf2f93673a2793ccf09c616',
            '69439f6799adc0ecebe5d8062ce0db5b','6b037a194ef4976eeb4803b62c4fb719','7040cd5b48783e42bf69ad8ae1130db5',
            '73a5736925da1fdb859ab54bfdedbfff','7419e251e895417a3a3ad7e47f688b29','774354380b368224856c600805762b03',
            '778606017ea378b29c4235662dd77a9b','781c7032ce133dd852b96ccd2d11e4bf','7f7464781c2b59b1979ed129ba5866ef',
            '8255666eee1ff0e1738dbd63d47ab6aa','83fd66c1795678c3116e9398298fe73a','84ee1eaf7070d3a9c2bca4e4bc041bf2',
            '87bc682f79ea761ddbdbb013050a1c5b','87c81ea150ff908dde537131cd2b8678','890b8e1488a926c3d61ca5cdf269c042',
            '898b48ced3f1e9dd37295d534c53a39d','8bb9b6b57afd916d9cbd154b58aee93a','8f497041c5cc920c1189c006b02be284',
            '8fe7afaf70fc15b77f5a4a45076db512','91042073096161389f78bcf12b373ac8','922c7f6d9b7ada7bf349b9ca9e0690c8',
            '927974d7535434e1dcf3c7cfba4b9493','94b51da11d9030efe8f0ae2a258b5249','958fcb3de326daceb8bca6f71c7f5f31',
            '974cabadc622c98bacb02e0c11cca489','97d6876c1ab25f3b1b157afea04ce085','991e202dbf8f3fa69c5bea40e4e22d32',
            '9ba083b4aeef838bb60273a1d85c51e8','9c83331eaf085e5226215fa8b0183264','a19c4a24fd629208c04e8da5226c18ec',
            'a2f272d11c96df42d2163777bc8b9530','a7bc2122f362b14549d93cc3f888211c','a91c38bf9a58e1f03b7d88d952183175',
            'ab2bc61274221eb593838366c4046d4e','b70b10fecd0dc4459b5f79fe1c27a066','ba04153243cd16383154dd03c3ab3d3d',
            'bb95450097bf3c9225491ab15f23bcbe','bbc54ea74865d7a669648770bec1a8ed','bbf942add1eb11d25de4bcd587255e8d',
            'bcd63981899e769ab4986af9cf68ecbc','bde50fe656168818a77249c148a49889','bdef2cfe5abcfbd00ec4e89bbdecdb4c',
            'be39412453a33d32d0cbc6fa2212e28f','bfcd1ac21134401a12e49a4d6be167a3','bfea8bd973b40144d41ecb96a463991a',
            'bff2ebb09d73c89167241250440c475e','c0adcc870ca075369537f7cd0877576a','c43e268913b76a1459b065476b86e2ff',
            'c45f458bc9561029985c973235627185','c4ba9f5f0b6ae3cfedc6bbc8a3b2c2a0','c4d2a7f360cd8f7b22cd7cf748b83ec7',
            'c8cec1bb3ccdb5e85d4a2068d81055b0','c9c7312b0816720c953a449dc3007635','cab79650794648699c676a3d19cdb009',
            'cb0166129d1c010d264776aef3b55510','ccc01717be747dd7d59a19ee79544b0a','cdb953820470775894008c4cb16c2c17',
            'd16aaa7f5eca772d64e756821630ec7f','d27c1300551cd5163d6f895adbba91cd','d42790b38a297d4f424f5c6503d99d62',
            'd5d7628be40c7cf6ebe8246a46f8038a','d7edbb8d7813c32bb838a90e7c485b2b','da853215664367774c410210e67d1028',
            'de9c1b2cf691af973ae00cc2d8e3ec15','e1ff6970a403794a37514b93e34a111f','e2283dc2db45396b4c6cce396168a8bf',
            'e2c6a76355fa923cd66d93359c3bd938','e38e043f8c26f5c4a6ec970bda188474','e414586b30c0b6d57a34ea69e0c30c0e',
            'e647db52770cc1af0cb377543b4eb303','e7567ff5caecf54f0ff222be914725dc','ea7455ba2d24883ea75b5b8213bb5569',
            'eb1c1a8c716fb70f25f745ebbc3b3b8f','eb1f2d920e97bb91ad6c9fdb809838fa','ef70587998eb7bc12c6466c36590a539',
            'effff673ae3df86f25970195d9fcf269','f5859bf3807ff3daa51967cce30a8d17','f967b6504968aaee43c2c6ac6ec3234a',
            'fc636cef2e6ea2fe9d9db02eeb673d11'] 

In [15]:
df1 = df[~df['patient-id'].isin(list_pid)]

In [16]:
# generate target
target = df1['outcome-Mds-fall-since-prior-assessment'].map({'None': 0, 'Yes': 1})
dff = df1.drop(['outcome-mds-fall-no-injury', 'outcome-mds-fall-minor-injury', 'outcome-mds-fall-major-injury', \
               'outcome-Mds-fall-since-prior-assessment', 'outcome-riskmaster-fall-incident'], axis=1)

In [17]:
# in sdd 0 means NA
for i in range(1, 6):
    #df.loc[df['psych_sdd_%d' % i] == 0, 'psych_sdd_%d' % i] = np.nan
    dff.loc[dff[f'psych_sdd_{i}'] == 0, f'psych_sdd_{i}'] = np.nan

In [18]:
# train test split
from sklearn.model_selection import GroupShuffleSplit
def train_test_split(df, target, groupby, radio, random_state):
    train_idx, test_idx = next(GroupShuffleSplit(train_size=radio, test_size=1-radio, random_state=random_state).split(df, target, df[groupby]))
    return df.iloc[train_idx], df.iloc[test_idx], target.iloc[train_idx], target.iloc[test_idx]

In [19]:
X_train, X_test, y_train, y_test = train_test_split(dff, target, 'patient-id', 0.7, 0)
print(f'Total number of patients: {len(df.groupby("patient-id"))}')
print(f'Number of patients in training set: {len(X_train.groupby("patient-id"))}')

Total number of patients: 4903
Number of patients in training set: 3332


In [20]:
# drop feature
def drop_features(df, f):
    df.drop(f, axis=1, inplace=True)
    
# one hot encoding
def one_hot_encoding(df, features, categories):
    df = df.copy()  # Make sure we're working on a copy
    for f, c in zip(features, categories):
        df[f] = pd.Categorical(df[f], categories=c)
    return pd.get_dummies(df, columns=features, prefix=features)

# compute date difference
def date_diff(start, end):
    return (pd.to_datetime(end) - pd.to_datetime(start)) / np.timedelta64(1, 'D')

# generate date diff features
def generate_period(df, f):
    df = df.copy()  # Ensure df is not a view
    for start, end, period in f:
        df[period] = date_diff(df[start], df[end])
        df.drop([start, end], axis=1, inplace=True)
    return df

# label encoding
def label_encoding(df, features, dictionaries):
    for f, dic in zip(features, dictionaries):
        df.loc[:, f] = df[f].map(dic)
        
# sin&cos transform
def col_transform(df, col, mmax, mmin, df_ref=None):
    if mmax <= mmin:
        # For test set, if max&min not set, use training set value
        if df_ref is not None: 
            mmax = df_ref[col].max()
            mmin = df_ref[col].min()
        else:
            mmax = df[col].max()
            mmin = df[col].min()

    # Ensure column is numeric
    numeric_col = pd.to_numeric(df[col], errors='coerce')
    
    # Calculate angle
    angle = 0.25 * np.pi * (numeric_col - mmin) / (mmax - mmin) + 0.125 * np.pi

    # Apply trigonometric functions
    return np.cos(angle), np.sin(angle)

def df_transform(df, f, df_ref=None):
    df = df.copy()  # Ensure df is not a view
    for col, mmax, mmin in f:
        cos_col, sin_col = col_transform(df, col, mmax, mmin, df_ref)
        df.loc[:, col + '_x'] = cos_col
        df.loc[:, col + '_y'] = sin_col
        df.fillna({col + '_x': 0}, inplace=True) # df[col + '_x'].fillna(0, inplace=True)
        df.fillna({col + '_y': 0}, inplace=True) #df[col + '_y'].fillna(0, inplace=True)
        df.drop(col, axis=1, inplace=True)
    return df

In [21]:
%%time
X_train = generate_period(X_train, f_period)
label_encoding(X_train, f_label, dic_label)
X_train_original = X_train.copy() # store value range
X_train = df_transform(X_train, f_transform)
X_train = one_hot_encoding(X_train, f_one_hot, cate_one_hot)
drop_features(X_train, f_drop)

CPU times: user 503 ms, sys: 13.3 ms, total: 516 ms
Wall time: 514 ms


In [22]:
X_train_use = X_train.copy()

In [23]:
# Display a scrollable DataFrame
HTML(X_train.to_html(notebook=True, max_rows=10, max_cols=200))

,patient-id,current-stay-days,cumulative-days-in-facility,cms-long-stay,mds-stay-trans-short-to-long,age,marital-status,mds-entered-from,mds-antianxiety-medication,mds-antidepressant-medication,mds-antipsychotic-medication,mds-antibiotic-medication,mds-anticoagulant-medication,mds-diuretic-medication,mds-hypnotic-medication,mds-behavioral-symptoms,mds-behavioral-symptoms-to-others,mds-dehydrated,mds-depression,mds-fever,mds-impaired-mobility,mds-impaired-transfer,mds-internal-bleeding,mds-pain-non-verbal,mds-pain-medication,mds-received-pain-tx-non-pharm,mds-received-prn-pain-medication,mds-excess-weight-loss,mds-malnutrition,mds-no-problem-conditions,mds-urinary-incontinence,mds-vomiting,pneumonia,uti,mdro,anemia,septicemia,constipation,wound,hyponatremia,hyperkalemia,embolisms,alzheimers,anxiety,depression,non-alz-dimentia,bipolar,parkinsons,psychosis,schizophrenia,seizure,aphasia,emphysema,arthritis,ashd,bph,cancer,cerebralpalsy,stroke,cirrhosis,comatose,diabetes,dysrhythmias,gerd,heart-failure,hemiplegia/hemiparesis,hepatitis,huntingtons,hyperlipidemia,hyperthyroidism,hypothyroidism,hypertension,hypotension,multiple-sclerosis,neurogenic-bladder,obstructive-uropathy,osteoporosis,paraplegia,ptsd,pvd,quadriplegia,thyroid-disorder,tourettes,transient-ischemic-attack,traumatic-brain-injury,tuberculosis,renal-failure,ws-antibiotic-anticoag-coexposure,ws-psychotropic-with-not-ordered-weight-loss,ws-diuretic-adl,ws-tramadol-antidepressant-coexposure,cns-drug-exposure,psychotropic-exposure,mds-conduct-staff-assessment-mental-status,mds-impaired-walk-in-room,mds-impaired-walk-in-corridor,mds-impaired-locomot-unit,mds-impaired-locomot-other,mds-pressure-ulcer-prsnt,conduct_bims,cam_inattention,cam_disorganized_thought,cam_altered_conc,cam_motor_retardation,acute_mental_change,conduct_pain_assmnt,mds-adl-scale,balance_while_standing,balance_while_walking,balance_turning_around,balance_toileting,balance_bed_to_chair,functlimit_rom_upper,functlimit_rom_lower,cane_or_crutch_past_7_days,walker_past_7_days,wheelchair_past_7_days,limb_prosthesis_past_7_days,no_listed_mobility_device,does_resident_wander,psych_sdd_final,atc_1,atc_2,atc_3,atc_4,atc_5,number_of_behavior_changes,episode-period,stay-period,mds-cognitive-scale_x,mds-cognitive-scale_y,mds-pain-freq-last-five-days_x,mds-pain-freq-last-five-days_y,mds-pain-intensity_x,mds-pain-intensity_y,Psychotropic-1-Period_x,Psychotropic-1-Period_y,Psychotropic-2-Period_x,Psychotropic-2-Period_y,Psychotropic-3-Period_x,Psychotropic-3-Period_y,Psychotropic-4-Period_x,Psychotropic-4-Period_y,Psychotropic-5-Period_x,Psychotropic-5-Period_y,Mds-fall-2-6-months-to-admission_Yes,Mds-fall-2-6-months-to-admission_No,Mds-fall-2-6-months-to-admission_Unable to answer,Mds-fall-last-month-to-admission_Yes,Mds-fall-last-month-to-admission_No,Mds-fall-last-month-to-admission_Unable to answer,mds-pain-last-five-days_Yes,mds-pain-last-five-days_No,mds-pain-last-five-days_Unable to answer,mds-delirium-scale_Yes,mds-delirium-scale_No,mds-long-term-memory-ok_Yes,mds-long-term-memory-ok_No,mds-short-term-memory-ok_Yes,mds-short-term-memory-ok_No,mds-staff-assess-pain_Yes,mds-staff-assess-pain_No,recent-start-other-fall-risk-rx_Yes,recent-start-other-fall-risk-rx_No,mds-pressure-ulcer-stage-1_Yes,mds-pressure-ulcer-stage-1_No,mds-pressure-ulcer-stage-2_Yes,mds-pressure-ulcer-stage-2_No,mds-pressure-ulcer-stage-3_Yes,mds-pressure-ulcer-stage-3_No,mds-pressure-ulcer-stage-4_Yes,mds-pressure-ulcer-stage-4_No,race_White,race_Black,race_Other,facility_Sugar Creek,facility_Heritage Place,facility_Canterbury Place,facility_Senaca Place,facility_Cranberry Place,gender_Female,gender_Male,mds-bims-summary-ranking_Intact or Moderately Intact,mds-bims-summary-ranking_Moderate Impairment,ws-meclizine-psych-coexposure_No,ws-meclizine-psych-coexposure_meclAndPsychStartSameR,ws-meclizine-psych-coexposure_meclPrecedesPsychR,ws-meclizine-psych-coexposure_psychPrecedesMeclR,ws-meclizine-psych-coexposure_meclAndPsychOlderStart
0,10353,

In [24]:
%%time
X_test = generate_period(X_test, f_period)
label_encoding(X_test, f_label, dic_label)
X_test = df_transform(X_test, f_transform, X_train_original)
X_test = one_hot_encoding(X_test, f_one_hot, cate_one_hot)
drop_features(X_test, f_drop)

CPU times: user 256 ms, sys: 712 µs, total: 257 ms
Wall time: 255 ms


In [25]:
X_test_use = X_test.copy()

In [26]:
X_train_use.shape

(8600, 189)

In [27]:
X_train_use.to_csv (r'export_simpleRNN_exp1_train_dataframe.csv', index = False, header=True)

In [28]:
X_test_use.shape

(3692, 189)

In [29]:
X_test_use.to_csv (r'export_simpleRNN_exp1_TEST_dataframe.csv', index = False, header=True)

**RNN - SimpleRNN**

In [30]:
# missing data describe
def missing_summary(df):
    missing_cnt = df.isna().sum()
    missing_data = pd.concat([missing_cnt, missing_cnt/df.shape[0]], axis=1, keys=['count', 'percentage'])
    return missing_data[missing_data['percentage'] != 0].sort_values(by='percentage', ascending=False)

In [31]:
idx = missing_summary(X_train_use).index.tolist()
X_train_use.drop(columns=idx, axis=1, inplace=True)
X_test_use.drop(columns=idx, axis=1, inplace=True)

In [32]:
def scaler_transform(X_train_use, X_test_use):    
    scaler = MinMaxScaler(feature_range=(0, 1)).fit(X_train_use.drop('patient-id', axis=1))
    X_train_transformed = pd.DataFrame(scaler.transform(X_train_use.drop('patient-id', axis=1)), \
                                       columns=X_train_use.columns[1:], index=X_train_use.index)                                   
    X_train_transformed['patient-id'] = X_train_use['patient-id']
    X_test_transformed = pd.DataFrame(scaler.transform(X_test_use.drop('patient-id', axis=1)), \
                                       columns=X_test_use.columns[1:], index=X_test_use.index) 
    X_test_transformed['patient-id'] = X_test_use['patient-id']
    return X_train_transformed, X_test_transformed

In [33]:
def add_previous_outcome(X, Y):
    X = X.copy()
    x1, x2 = [], []
    pids = X['patient-id'].unique()
    for pid in pids:
        df = Y[X[X['patient-id']==pid].index]
        if(df.shape[0] == 1):
            x1.append(pd.Series([0]))
            x2.append(pd.Series([0]))
        else:
            # input 1=(1, 0), 0=(0, 1), NA=(0, 0)
            x1.append(pd.Series([0]))
            x1.append(df[:-1])
            x2.append(pd.Series([0]))
            x2.append(df[:-1].map({0: 1, 1: 0}))
    X['input_outcome_x'] = pd.Series(pd.concat(x1, ignore_index=True).tolist(), index=X.index)
    X['input_outcome_y'] = pd.Series(pd.concat(x2, ignore_index=True).tolist(), index=X.index)
    return X

In [34]:
## This step converts the outcome to a point in x,y coordinates where fall (1) = (1,0); no fall (0) = (0,1); and NA = (0,0) 
X_train_use = add_previous_outcome(X_train_use, y_train)
X_test_use = add_previous_outcome(X_test_use, y_test)

## This step transforms the data to range between 0 and 1
X_train_transformed, X_test_transformed = scaler_transform(X_train_use, X_test_use)

In [35]:
X_train_transformed.shape

(8600, 102)

In [36]:
X_test_transformed.shape

(3692, 102)

In [37]:
non_numeric_columns = X_train_transformed.select_dtypes(exclude=[np.number]).columns
print("Non-numeric columns:", non_numeric_columns)

Non-numeric columns: Index(['patient-id'], dtype='object')


In [38]:
series = X_train_transformed.drop('patient-id', axis=1).mean(axis=0)
HTML(series.to_frame().to_html(notebook=True, max_rows=200, max_cols=2))

,0
current-stay-days,0.096641
cumulative-days-in-facility,0.136353
cms-long-stay,0.330581
mds-stay-trans-short-to-long,0.160349
age,0.690848
mds-antianxiety-medication,0.130581
mds-antidepressant-medication,0.504884
mds-antipsychotic-medication,0.147442
mds-antibiotic-medication,0.224767
mds-anticoagulant-medication,0.309419


In [39]:
series = X_train_transformed.drop('patient-id', axis=1).std(axis=0)
HTML(series.to_frame().to_html(notebook=True, max_rows=200, max_cols=2))

,0
current-stay-days,0.165251
cumulative-days-in-facility,0.201916
cms-long-stay,0.470450
mds-stay-trans-short-to-long,0.366951
age,0.150383
mds-antianxiety-medication,0.336962
mds-antidepressant-medication,0.500005
mds-antipsychotic-medication,0.354566
mds-antibiotic-medication,0.417453
mds-anticoagulant-medication,0.462281


In [40]:
series = X_train_transformed.drop('patient-id', axis=1).min(axis=0)
HTML(series.to_frame().to_html(notebook=True, max_rows=200, max_cols=2))

,0
current-stay-days,0.0
cumulative-days-in-facility,0.0
cms-long-stay,0.0
mds-stay-trans-short-to-long,0.0
age,0.0
mds-antianxiety-medication,0.0
mds-antidepressant-medication,0.0
mds-antipsychotic-medication,0.0
mds-antibiotic-medication,0.0
mds-anticoagulant-medication,0.0


In [41]:
series = X_train_transformed.drop('patient-id', axis=1).max(axis=0)
HTML(series.to_frame().to_html(notebook=True, max_rows=200, max_cols=2))

,0
current-stay-days,1.0
cumulative-days-in-facility,1.0
cms-long-stay,1.0
mds-stay-trans-short-to-long,1.0
age,1.0
mds-antianxiety-medication,1.0
mds-antidepressant-medication,1.0
mds-antipsychotic-medication,1.0
mds-antibiotic-medication,1.0
mds-anticoagulant-medication,1.0


In [42]:
X_train_transformed.to_csv (r'exp1_train_transformed_df.csv', index = False, header=True)

In [43]:
X_train_use.shape

(8600, 102)

In [44]:
X_test_use.shape

(3692, 102)

In [45]:
X_train_use.head()

,patient-id,current-stay-days,cumulative-days-in-facility,cms-long-stay,mds-stay-trans-short-to-long,age,mds-antianxiety-medication,mds-antidepressant-medication,mds-antipsychotic-medication,mds-antibiotic-medication,...,gender_Male,mds-bims-summary-ranking_Intact or Moderately Intact,mds-bims-summary-ranking_Moderate Impairment,ws-meclizine-psych-coexposure_No,ws-meclizine-psych-coexposure_meclAndPsychStartSameR,ws-meclizine-psych-coexposure_meclPrecedesPsychR,ws-meclizine-psych-coexposure_psychPrecedesMeclR,ws-meclizine-psych-coexposure_meclAndPsychOlderStart,input_outcome_x,input_outcome_y
0,10353,7,7,0,0,67,1,1,0,0,...,False,True,False,True,False,False,False,False,0.0,0.0
1,10364,7,7,0,1,84,0,0,1,1,...,False,True,False,True,False,False,False,False,0.0,0.0
2,10379,7,7,0,0,84,0,1,0,0,...,False,True,False,True,False,False,False,False,0.0,0.0
5,5000,7,7,0,0,90,0,1,1,0,...,False,True,False,True,False,False,False,False,0.0,0.0
6,5003,7,7,0,1,90,0,1,1,1,...,True,True,False,True,False,False,False,False,0.0,0.0


In [46]:
X_train_transformed.head()

,current-stay-days,cumulative-days-in-facility,cms-long-stay,mds-stay-trans-short-to-long,age,mds-antianxiety-medication,mds-antidepressant-medication,mds-antipsychotic-medication,mds-antibiotic-medication,mds-anticoagulant-medication,...,mds-bims-summary-ranking_Intact or Moderately Intact,mds-bims-summary-ranking_Moderate Impairment,ws-meclizine-psych-coexposure_No,ws-meclizine-psych-coexposure_meclAndPsychStartSameR,ws-meclizine-psych-coexposure_meclPrecedesPsychR,ws-meclizine-psych-coexposure_psychPrecedesMeclR,ws-meclizine-psych-coexposure_meclAndPsychOlderStart,input_outcome_x,input_outcome_y,patient-id
0,0.007423,0.006342,0.0,0.0,0.558140,1.0,1.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,10353
1,0.007423,0.006342,0.0,1.0,0.755814,0.0,0.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,10364
2,0.007423,0.006342,0.0,0.0,0.755814,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,10379
5,0.007423,0.006342,0.0,0.0,0.825581,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,5000
6,0.007423,0.006342,0.0,1.0,0.825581,0.0,1.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,5003


In [47]:
X_train_transformed.shape

(8600, 102)

In [48]:
def generate_nested_list(X, Y):
    x_list, y_list = [], [] 
    pids = X['patient-id'].unique()
    for pid in pids:
        df = X[X['patient-id']==pid]
        recs = [record for record in df.drop('patient-id', axis=1).values]
        if len(recs) > 1:
            y_list.append(Y[df.index].values)
            x_list.append(recs)
        #x_list.append([record for record in df.drop('patient-id', axis=1).values])
    return x_list, y_list

def mygenerator(x_list, y_list=None):
    if(y_list is not None):
        while True:
            for x, y in zip(x_list, y_list):
                yield np.array(x).reshape((len(x), 1, x[0].shape[0])), y
    else:
        while True:
            for x in x_list:
                yield np.array(x).reshape((len(x), 1, x[0].shape[0])) 

def plot_roc_curve(fpr, tpr):
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', label='ROC curve (area = %0.2f)' % auc(fpr, tpr))
    plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc="lower right")
    plt.show()
    
def plot_history(history, loss=None, val_loss=None):
    if(history is not None):
        loss = history.history['loss']
    plt.figure()
    plt.plot(loss)
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    if('val_loss' in history.history):
        val_loss = history.history['val_loss']
        plt.plot(val_loss)
    plt.legend(['train', 'test'], loc='upper right')
    plt.show()

In [49]:
def pad_features(X, maxlen, padding_value=0):
    """
    Pads a list of feature sequences to the specified maximum sequence length.
    
    Args:
        X (list of lists or tensors): A list of sequences with shape (sequence_length, feature_dim).
        maxlen (int): The maximum length to pad the sequences to.
        padding_value (float): The value to use for padding (default: 0).
    
    Returns:
        torch.Tensor: A tensor of shape (batch_size, maxlen, feature_dim).
    """
    # Convert all sequences to tensors
    sequences = [torch.tensor(seq, dtype=torch.float32) for seq in X]
    # Truncate sequences to maxlen
    truncated = [seq[:maxlen] for seq in sequences]
    # Pad sequences
    padded = pad_sequence(truncated, batch_first=True, padding_value=padding_value)
    
    # Ensure the result has the correct number of dimensions
    if padded.shape[1] < maxlen:
        extra_padding = torch.zeros(
            (padded.size(0), maxlen - padded.size(1), padded.size(2)), 
            dtype=padded.dtype, 
            device=padded.device
        )
        padded = torch.cat([padded, extra_padding], dim=1)
    
    return padded

def pad_labels(y, maxlen, padding_value=0):
    """
    Pads or broadcasts scalar or 1D labels to the specified maximum sequence length.
    """
    # Handle scalar or 1D labels
    if all(isinstance(label, (int, float)) or len(label) == 1 for label in y):
        labels = torch.tensor([[label] * maxlen for label in y], dtype=torch.float32)
    else:
        # Convert to tensors and pad
        padded_labels = []
        for label in y:
            label_tensor = torch.tensor(label, dtype=torch.float32)
            # Truncate or pad each label
            if len(label_tensor) < maxlen:
                padded = torch.cat([label_tensor, torch.full((maxlen - len(label_tensor),), padding_value)])
            else:
                padded = label_tensor[:maxlen]
            padded_labels.append(padded)
        labels = torch.stack(padded_labels)

    # Add feature dimension
    return labels.unsqueeze(-1)


def pad_all_data(X_train, y_train, X_test, y_test, maxlen, padding_value=0):
    """
    Pads all feature and label data for training and testing.
    
    Args:
        X_train, y_train, X_test, y_test: Input data (features and labels).
        maxlen: Maximum sequence length to pad to.
        padding_value: Padding value (default: 0).
    
    Returns:
        Tuple[torch.Tensor]: Padded versions of the input data.
    """
    X_train_pad = pad_features(X_train, maxlen, padding_value)
    y_train_pad = pad_labels(y_train, maxlen, padding_value)
    X_test_pad = pad_features(X_test, maxlen, padding_value)
    y_test_pad = pad_labels(y_test, maxlen, padding_value)
    return X_train_pad, y_train_pad, X_test_pad, y_test_pad


In [50]:
def auroc(y_predict, y_true, plot=False):
    fpr, tpr, thresholds = roc_curve(y_true, y_predict, pos_label=1)    
    if(plot):
        plot_roc_curve(fpr, tpr)
    return auc(fpr, tpr)


def find_best_f1(y_predict, y_true):
    fpr, tpr, thresholds = roc_curve(y_true, y_predict, pos_label=1) 
    max_f1, th = 0, 0
    for threshold in thresholds:
        f1 = f1_score(y_true, y_predict > threshold)
        if(f1 > max_f1):
            max_f1 = f1
            th = threshold
    return max_f1, th

def other_metrics(y_predict, y_true, threshold):
    y_threshold = y_predict > threshold
    tn, fp, fn, tp = confusion_matrix(y_true, y_threshold).ravel()
    return precision_score(y_true, y_threshold), recall_score(y_true, y_threshold),\
            accuracy_score(y_true, y_threshold), float(tn) / (tn + fp)


In [51]:
# %%time
# shape: (num_of_patients, num_of_records, num_of_features)
X_train_list, y_train_list = generate_nested_list(X_train_transformed, y_train)
X_test_list, y_test_list = generate_nested_list(X_test_transformed, y_test)

In [52]:
X_train_lengths = [len(x) for x in X_train_list]
X_test_lengths = [len(x) for x in X_test_list]

In [53]:
X_train_lengths[0:30]

[2,
 3,
 7,
 7,
 8,
 7,
 8,
 10,
 7,
 4,
 9,
 7,
 2,
 2,
 2,
 2,
 2,
 4,
 5,
 4,
 4,
 5,
 2,
 4,
 11,
 5,
 4,
 12,
 15,
 2]

In [54]:
X_test_lengths[0:30]

[3,
 5,
 11,
 2,
 2,
 2,
 14,
 11,
 5,
 4,
 9,
 2,
 2,
 11,
 3,
 2,
 13,
 7,
 3,
 2,
 5,
 8,
 13,
 2,
 2,
 2,
 2,
 2,
 8,
 2]

In [55]:
X_train_list[4]

[array([0.11240721, 0.11099366, 1.        , 0.        , 0.79069767,
        1.        , 1.        , 1.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 1.        , 1.        , 0.        ,
        0.        , 0.        , 1.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.375     , 0.375     ,
        0.        , 0.        , 0.        , 0.        , 0.60714286,
        0.03125   , 0.        , 0.8125    , 0.79069767, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.85793617, 0.65993818, 0.85248798, 0.66696111, 0.77943396,
        0.75103633, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 1.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.  

In [56]:
max_timestamps = df.groupby(['patient-id']).size().max()
print(max_timestamps)

20


In [57]:
n_features = X_train_list[0][0].shape[0]
print(n_features)

101


In [58]:
class MaskedRNNTimeDist(nn.Module):
    def __init__(self, hidden_dim, dropout_rate, input_dim, output_dim, seq_length):
        super(MaskedRNNTimeDist, self).__init__()
        self.hidden_dim = hidden_dim
        self.seq_length = seq_length
        #self.rnn = nn.RNN(input_dim, hidden_dim, batch_first=True)
        self.rnn = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)
        self.time_distributed = nn.Linear(hidden_dim, output_dim)  # Time-distributed dense layer
        self.activation = nn.Tanh()  # Tanh activation

    def forward(self, x, lengths):
        # Create mask for non-zero values
        #mask = (x != 0).float()
        #x = x * mask  # Apply mask
        packed_input = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        packed_output, _ = self.rnn(packed_input)
        output, _ = pad_packed_sequence(packed_output, batch_first=True)
        # Unpack the sequence
        output, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=x.size(1))

        # Time-distributed dense layer
        output = self.time_distributed(output)  # Apply dense layer across time steps

        # Clamp outputs before applying the activation
        #output = torch.clamp(output, min=1e-6, max=1-1e-6)  # Numerical stability safeguard

        # Apply sigmoid activation
        output = self.activation(output)  
        
        return output  # Final shape: (batch_size, seq_length, output_dim)

def build_model(units, dropout_ratio, n_timesteps, n_dimensions):
    hidden_dim = units
    dropout_rate = dropout_ratio
    input_dim = n_dimensions
    output_dim = 1  # Output size for binary classification
    
    # Move model to GPU
    device = torch.device("cuda")
    
    model = MaskedRNNTimeDist(
        hidden_dim=hidden_dim,
        dropout_rate=dropout_rate,
        input_dim=input_dim,
        output_dim=output_dim,
        seq_length=n_timesteps
    ).to(device)
    
    return model


In [59]:
def custom_bce_loss_with_logits(outputs, targets, pos_weight=None):
    predictions = torch.sigmoid(outputs)
    # Add small epsilon to avoid log(0)
    epsilon = 1e-8
    predictions = torch.clamp(predictions, epsilon, 1 - epsilon)

    if pos_weight is not None:
        pos_weight = pos_weight.to(outputs.device)
        loss = - (pos_weight * targets * torch.log(predictions) +
                  (1 - targets) * torch.log(1 - predictions))
    else:
        loss = - (targets * torch.log(predictions) +
                  (1 - targets) * torch.log(1 - predictions))
    
    return loss.mean()


In [60]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)  # Probability for correct class
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [61]:
def train_model(model, X_train, lengths, y_train, pos_weight, epochs, batch_size, optimizer, device):
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

    X_train = X_train.to(device)
    y_train = y_train.to(device)
    print(f'train_model: moved X_train and y_train to {str(device)}')

    # Debug
    print(f"X_train min={X_train.min().item()}, max={X_train.max().item()}, mean={X_train.mean().item()}")
    
    metrics = {"loss": [], "accuracy": []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct_predictions = 0
        total_predictions = 0

        perm = torch.randperm(len(X_train))
        X_train = X_train[perm]
        y_train = y_train[perm]
      
        print(f"train_model: Running epoch {epoch + 1}/{epochs}")
        for i in range(0, len(X_train), batch_size):
            x_batch = X_train[i:i+batch_size]
            y_batch = y_train[i:i+batch_size]
            lengths_batch = lengths[i:i+batch_size]
            
            # Debug output to confirm x_batch is valid
            print(f"x_batch min={x_batch.min().item()}, max={x_batch.max().item()}, mean={x_batch.mean().item()}")
            print(f"x_batch lengths: {lengths_batch}")

            outputs = model(x_batch, lengths_batch)

            # Replace NaNs and infinities with 0
            outputs = torch.nan_to_num(outputs, nan=0.0, posinf=1.0, neginf=-1.0)

            # Debug hidden states and outputs
            print(f"Step {i}: outputs min={outputs.min().item()}, max={outputs.max().item()}")
            print(f"Step {i}: outputs shape={outputs.shape}")
            
            # Check for invalid values
            if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                print("Outputs contain invalid values!")
                print(f"outputs: {outputs}")
                return metrics
            if torch.isnan(y_batch).any() or torch.isinf(y_batch).any():
                print("y_batch contains invalid values!")
                print(f"y_batch: {y_batch}")
                return metrics
            
            # Compute loss
            try:                
                #loss = custom_bce_loss_with_logits(outputs, y_batch, pos_weight=pos_weight) # much greater loss but does not decline
                #criterion = FocalLoss(alpha=0.25, gamma=2)  # very stable moderate loss but does not decline
                criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
                loss = criterion(outputs, y_batch)
                print(f"Step {i}: Loss={loss.item()}")  # Log the loss value                
            except RuntimeError as e:
                print(f"RuntimeError during loss computation: {e}")
                return metrics
            
            optimizer.zero_grad()
            loss.backward()
            # Track gradients
            for name, param in model.named_parameters():
                if param.grad is not None:
                    print(f"Gradient for {name}: min={param.grad.min().item()}, max={param.grad.max().item()}")
                else:
                    print(f"Gradient for {name}: None")
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Clip gradients
            optimizer.step()

            running_loss += loss.item()
            predictions = (outputs > 0.5).float()
            correct_predictions += (predictions == y_batch).sum().item()
            total_predictions += y_batch.numel()

        epoch_loss = running_loss / ((len(X_train) + batch_size - 1) // batch_size)
        epoch_accuracy = correct_predictions / total_predictions

        print(f"Epoch {epoch + 1} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")
        metrics["loss"].append(epoch_loss)
        metrics["accuracy"].append(epoch_accuracy)

        scheduler.step()
        
    return metrics

In [62]:
## the data is padded so that the final shape is as follows with the outcome data array having the same
## shape but only one value
#  array([[patient 1 data record 1],  
#         [patient 1 data record 2],
#         ...
#         [patient 1 data record 19],
#        ],
#        [[patient 2 data record 1],  
#         [patient 2 data record 2],
#         ...
#         [patient 2 data record 19],
#        ],
#        ...
#       )
# 
# Apply padding to all data
X_train_pad, y_train_pad, X_test_pad, y_test_pad = pad_all_data(
    X_train_list, y_train_list, X_test_list, y_test_list, max_timestamps
)

# Verify shapes
# Print results
print("X_train_pad shape:", X_train_pad.shape)
print("y_train_pad shape:", y_train_pad.shape)
print("X_test_pad shape:", X_test_pad.shape)
print("y_test_pad shape:", y_test_pad.shape)


X_train_pad shape: torch.Size([1657, 20, 101])
y_train_pad shape: torch.Size([1657, 20, 1])
X_test_pad shape: torch.Size([706, 20, 101])
y_test_pad shape: torch.Size([706, 20, 1])


/scratch/slurm-3021/ipykernel_1764/2925676546.py:14: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647348947/work/torch/csrc/utils/tensor_new.cpp:278.)
  sequences = [torch.tensor(seq, dtype=torch.float32) for seq in X]


In [63]:
# Replace NaNs and infinities with 0
X_train_pad = torch.nan_to_num(X_train_pad)
X_test_pad = torch.nan_to_num(X_test_pad)

In [64]:
print("Are there NaNs in X_train_pad?", torch.isnan(X_train_pad).any().item())
print("Are there infinities in X_train_pad?", torch.isinf(X_train_pad).any().item())
print("Are there NaNs in X_test_pad?", torch.isnan(X_test_pad).any().item())
print("Are there infinities in X_test_pad?", torch.isinf(X_test_pad).any().item())

Are there NaNs in X_train_pad? False
Are there infinities in X_train_pad? False
Are there NaNs in X_test_pad? False
Are there infinities in X_test_pad? False


In [65]:
print("Input shape:", X_train_pad.shape, "dtype:", X_train_pad.dtype)
print("Target shape:", y_train_pad.shape, "dtype:", y_train_pad.dtype)

print("Train values shape:", X_test_pad.shape, "dtype:", X_test_pad.dtype)
print("Target values shape:", y_test_pad.shape, "dtype:", y_test_pad.dtype)

Input shape: torch.Size([1657, 20, 101]) dtype: torch.float32
Target shape: torch.Size([1657, 20, 1]) dtype: torch.float32
Train values shape: torch.Size([706, 20, 101]) dtype: torch.float32
Target values shape: torch.Size([706, 20, 1]) dtype: torch.float32


In [66]:
# Replace NaNs and infinities with 0
y_train_pad = torch.nan_to_num(y_train_pad)
y_test_pad = torch.nan_to_num(y_test_pad)

In [67]:
print("Are there NaNs in y_train_pad?", torch.isnan(y_train_pad).any().item())
print("Are there infinities in y_train_pad?", torch.isinf(y_train_pad).any().item())
print("Are there NaNs in y_test_pad?", torch.isnan(y_test_pad).any().item())
print("Are there infinities in y_test_pad?", torch.isinf(y_test_pad).any().item())

Are there NaNs in y_train_pad? False
Are there infinities in y_train_pad? False
Are there NaNs in y_test_pad? False
Are there infinities in y_test_pad? False


In [68]:
# Check if all elements in y_train_pad are zero
is_all_zero = torch.all(y_train_pad == 0).item()

# Print the result
if is_all_zero:
    print("The array y_train_pad is entirely composed of 0s.")
else:
    print("The array y_train_pad contains non-zero elements.")

The array y_train_pad contains non-zero elements.


In [69]:
# Check if all elements in y_test_pad are zero
is_all_zero = torch.all(y_test_pad == 0).item()

# Print the result
if is_all_zero:
    print("The array y_test_pad is entirely composed of 0s.")
else:
    print("The array y_test_pad contains non-zero elements.")

The array y_test_pad contains non-zero elements.


In [70]:
for elt in [(y_train_pad,'y_train_pad'), (y_test_pad,'y_test_pad')]:
    # Check which vectors have at least one non-zero value
    non_zero_vectors = torch.any(elt[0] != 0, dim=(1, 2))

    # Count the number of such vectors
    num_non_zero_vectors = torch.sum(non_zero_vectors).item()

    # Print the result
    print(f"Number of vectors in {elt[1]} with at least one non-zero value: {num_non_zero_vectors}")

    # Calculate and print the percentage of non-zero elements
    total_non_zero_elements = torch.sum(elt[0] != 0).item()
    total_elements = elt[0].numel()
    percentage_non_zero = (total_non_zero_elements / total_elements) * 100
    print(f"Percentage of non-zero samples: {percentage_non_zero:.2f}%")


Number of vectors in y_train_pad with at least one non-zero value: 577
Percentage of non-zero samples: 3.68%
Number of vectors in y_test_pad with at least one non-zero value: 250
Percentage of non-zero samples: 3.90%


In [71]:
print("y_train_pad min:", y_train_pad.min().item())
print("y_train_pad max:", y_train_pad.max().item())
print("y_test_pad min:", y_train_pad.min().item())
print("y_test_pad max:", y_train_pad.max().item())

y_train_pad min: 0.0
y_train_pad max: 1.0
y_test_pad min: 0.0
y_test_pad max: 1.0


In [72]:
assert torch.all((y_train_pad >= 0) & (y_train_pad <= 1)), "Target values must be in range [0, 1]"

In [73]:
max_timestamps

20

In [74]:
n_features

101

In [75]:
### A single run w/out cross-validation
# with previous outcome added to the features
# outcome is added before scaling
units = 64
dropout_ratio = 0.1
n_timesteps = max_timestamps
n_dimensions = n_features

def initialize_weights(model):
    for name, param in model.named_parameters():
        if 'weight' in name:
            nn.init.xavier_uniform_(param)  # Xavier initialization
        elif 'bias' in name:
            nn.init.zeros_(param)

# Step 1: Delete the model
#del model

# Step 2: Clear GPU memory if necessary
#torch.cuda.empty_cache()

# Instantiate model and initialize weights
model = build_model(units, dropout_ratio, n_timesteps, n_dimensions)
initialize_weights(model)

# Debug to confirm initialization worked
for name, param in model.named_parameters():
    if 'weight' in name:
        print(f"Weight {name}: min={param.min().item()}, max={param.max().item()}, mean={param.mean().item()}")
    elif 'bias' in name:
        print(f"Bias {name}: min={param.min().item()}, max={param.max().item()}, mean={param.mean().item()}")


Weight rnn.weight_ih_l0: min=-0.12963928282260895, max=0.12963098287582397, mean=-0.0004506936529651284
Weight rnn.weight_hh_l0: min=-0.136885404586792, max=0.13692118227481842, mean=0.000557631254196167
Bias rnn.bias_ih_l0: min=0.0, max=0.0, mean=0.0
Bias rnn.bias_hh_l0: min=0.0, max=0.0, mean=0.0
Weight time_distributed.weight: min=-0.3021027445793152, max=0.3018891215324402, mean=-0.025271473452448845
Bias time_distributed.bias: min=0.0, max=0.0, mean=0.0


In [76]:
# Account for class balance
positive_class_ratio = 0.20
pos_weight_value = 1 / positive_class_ratio
positive_weight = torch.tensor([pos_weight_value], device="cpu")  # Ensure on CPU

# Training parameters
epochs = 5
batch_size = 32
device = torch.device("cuda")
#model = model.to(device)  # Ensure model is on the correct device

# Debug model parameters
#for name, param in model.named_parameters():
#    print(f"{name}: requires_grad={param.requires_grad}, device={param.device}")

# Optimizer
optimizer = torch.optim.RMSprop(model.parameters(), lr=5e-4)
#optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
#optimizer = torch.optim.Adagrad(model.parameters(), lr=1e-3)

# Train the model
metrics = train_model(model, X_train_pad, X_train_lengths, y_train_pad, positive_weight, epochs, batch_size, optimizer, device)

# Analyze metrics
print("Training Metrics:", metrics)

train_model: moved X_train and y_train to cuda
X_train min=0.0, max=1.0, mean=0.04512423276901245
train_model: Running epoch 1/5
x_batch min=0.0, max=1.0, mean=0.05445093289017677
x_batch lengths: [2, 3, 7, 7, 8, 7, 8, 10, 7, 4, 9, 7, 2, 2, 2, 2, 2, 4, 5, 4, 4, 5, 2, 4, 11, 5, 4, 12, 15, 2, 5, 6]
Step 0: outputs min=-0.25377723574638367, max=0.18575811386108398
Step 0: outputs shape=torch.Size([32, 20, 1])
Step 0: Loss=0.7871457934379578
Gradient for rnn.weight_ih_l0: min=-0.0038996688090264797, max=0.0033205896615982056
Gradient for rnn.weight_hh_l0: min=-0.001062872470356524, max=0.0010529820574447513
Gradient for rnn.bias_ih_l0: min=-0.009736328385770321, max=0.008571147918701172
Gradient for rnn.bias_hh_l0: min=-0.009736328385770321, max=0.008571147918701172
Gradient for time_distributed.weight: min=-0.005152238067239523, max=0.006035747937858105
Gradient for time_distributed.bias: min=0.39108362793922424, max=0.39108362793922424
x_batch min=0.0, max=1.0, mean=0.04944596439599991
x

In [81]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

def evaluate_model(model, X, y, lengths, batch_size, device):
    """
    Evaluate the model on a given dataset and compute metrics.
    
    Args:
        model: The trained PyTorch model.
        X: Input data (torch.Tensor).
        y: Ground truth labels (torch.Tensor).
        lengths: Lengths of sequences for packing (list).
        batch_size: Batch size for evaluation.
        device: Device ('cuda' or 'cpu').
    
    Returns:
        dict: Dictionary containing precision, recall, F1, AUROC, and AUPR scores.
    """
    model.eval()  # Set the model to evaluation mode
    X = X.to(device)
    y = y.to(device)
    
    y_true = []
    y_pred = []
    y_probs = []
    
    with torch.no_grad():  # No gradient computation for evaluation
        for i in range(0, len(X), batch_size):
            x_batch = X[i:i+batch_size]
            y_batch = y[i:i+batch_size]
            lengths_batch = lengths[i:i+batch_size]
            
            # Pack the batch and get predictions
            #packed_x = torch.nn.utils.rnn.pack_padded_sequence(x_batch, lengths_batch, batch_first=True, enforce_sorted=False)
            #outputs = model(packed_x, lengths_batch)
            outputs = model(x_batch, lengths_batch)
            
            # Apply sigmoid to get probabilities
            probs = torch.sigmoid(outputs)
            y_probs.append(probs.cpu().numpy())
            
            # Get binary predictions
            preds = (probs > 0.5).float()
            y_pred.append(preds.cpu().numpy())
            y_true.append(y_batch.cpu().numpy())
    
    # Concatenate results
    y_true = np.concatenate(y_true).ravel()
    y_pred = np.concatenate(y_pred).ravel()
    y_probs = np.concatenate(y_probs).ravel()
    
    # Compute metrics
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auroc = roc_auc_score(y_true, y_probs)
    aupr = average_precision_score(y_true, y_probs)
    
    metrics = {
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "AUROC": auroc,
        "AUPR": aupr
    }
    return metrics


In [82]:
# Evaluate on training set
train_metrics = evaluate_model(model, X_train_pad, y_train_pad, X_train_lengths, batch_size, device)
print("Training Metrics:")
for metric, value in train_metrics.items():
    print(f"{metric}: {value:.4f}")

# Evaluate on test set
test_metrics = evaluate_model(model, X_test_pad, y_test_pad, X_test_lengths, batch_size, device)
print("\nTest Metrics:")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")


Training Metrics:
Precision: 0.3039
Recall: 0.7217
F1 Score: 0.4277
AUROC: 0.7602
AUPR: 0.3081

Test Metrics:
Precision: 0.3041
Recall: 0.7127
F1 Score: 0.4263
AUROC: 0.7521
AUPR: 0.3710


# See conversation in chat 'rnn homework'

https://chatgpt.com/share/67423b46-e048-800d-909f-a570ba27cff3 

Section: Suggestions for Improvement